In [ ]:
# !pip install polars
# !pip install --upgrade pandas-gbq
# !pip install --upgrade pandas google-cloud-bigquery
# !pip install "numpy<2.0.0"
# !pip install --upgrade fsspec

In [ ]:
import os
import json
import re
import subprocess
import numpy as np
import pandas as pd
import pandas_gbq
import polars as pl
from google.cloud import bigquery
import pandas_gbq
import warnings
import matplotlib.pyplot as plt
from scipy.stats import norm

In [ ]:
#setup of variables
def wb(*args):
    """Run a wb command and return parsed JSON."""
    cmd = ["wb", *args, "--format=json"]
    result = subprocess.check_output(cmd, text=True)
    return json.loads(result)
 
 
# Get workspace info
workspace = wb("workspace", "describe")
GOOGLE_CLOUD_PROJECT = workspace["googleProjectId"]
 
# Get resources
resources = wb("resource", "list")
 
# WORKSPACE_BUCKET
bucket_resources = [
    r for r in resources
    if r.get("resourceType") == "GCS_BUCKET"
    and "practical_considerations_bucket" in r.get("id", "")
    and "temporary" not in r.get("id", "")
]
 
if not bucket_resources:
    raise ValueError("No matching bucket found")
 
WORKSPACE_BUCKET = f"gs://{bucket_resources[0]['bucketName']}"
 
# WORKSPACE_CDR
bq_resources = [
    r for r in resources
    if r.get("resourceType") in {"BQ_DATASET", "BIGQUERY_DATASET"}
]
 
cdr_resources = [
    r for r in bq_resources
    if re.match(r"^C\d{4}Q\d+R\d+$", r.get("datasetId", ""))
]
 
if not cdr_resources:
    raise ValueError("No matching CDR dataset found")
 
WORKSPACE_CDR = (
    f"{cdr_resources[0]['projectId']}."
    f"{cdr_resources[0]['datasetId']}"
)


In [ ]:
#get variables
version = WORKSPACE_CDR
bucket = WORKSPACE_BUCKET
cohort = "allofus"

In [ ]:
os.environ["WORKSPACE_CDR"] = version

In [ ]:
google_project_id = %env GOOGLE_CLOUD_PROJECT

In [ ]:
client = bigquery.Client(project=google_project_id)

def download_data(query):
    return client.query(query).to_dataframe()

In [ ]:
#set working directory to be within mounted bucket
#this means that reading and writing will now directly read from and write to the bucket
os.chdir("/home/dataproc/workspace/practical_considerations_bucket/mhdata")

In [ ]:
query = f"""
SELECT  person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        -- Add sex_at_birth columns
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        -- Added measurement columns here:
        meas.measurement_source_concept_id,
        meas.value_as_number,
        meas.measurement_datetime,
FROM `{WORKSPACE_CDR}.person` person 
-- New Join to get the actual measurement data
INNER JOIN `{WORKSPACE_CDR}.measurement` meas 
            ON person.person_id = meas.person_id
LEFT JOIN  `{WORKSPACE_CDR}.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
LEFT JOIN `{WORKSPACE_CDR}.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
LEFT JOIN `{WORKSPACE_CDR}.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
LEFT JOIN `{WORKSPACE_CDR}.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
LEFT JOIN `{WORKSPACE_CDR}.concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
WHERE meas.measurement_source_concept_id IN (903133)
"""

person_df = download_data(query)


In [ ]:
print(f"N participants: {len(person_df)}")

In [ ]:
# Rename value_as_number to height_cm
person_df = person_df.rename(columns={"value_as_number": "height_cm"})

# Compute median height per person, keeping sex_at_birth
height_df = (
    person_df.groupby(["person_id", "sex_at_birth"])["height_cm"]
    .median()
    .reset_index(name="median_height")
)

# Confirm no duplicate person_id
assert height_df["person_id"].is_unique, "Duplicate person_id found!"

In [ ]:
prs_df = pl.read_csv(f'{bucket}/mhdata/Height_output/score/Height_final_scores.csv')

In [ ]:
prs_df = prs_df.rename({"sample_id": "person_id"})

In [ ]:
prs_df = prs_df.with_columns(
    ((pl.col("sum_weights") - pl.col("sum_weights").mean()) / pl.col("sum_weights").std()).alias("sum_weights_scaled")
)

In [ ]:
prs_df = prs_df.to_pandas()

prs_df = prs_df.sort_values("sum_weights")
prs_df["sum_weights_decile"] = pd.qcut(prs_df["sum_weights"], 10, labels=False) + 1

In [ ]:
prs_df["person_id"] = prs_df["person_id"].astype("int64")

In [ ]:
df_merged = height_df.merge(prs_df, on="person_id", how="left")

In [ ]:
df_merged = df_merged[df_merged["sex_at_birth"].isin(["Male", "Female"])]

In [ ]:
correlation_SR = df_merged[["median_height", "sum_weights"]].dropna().corr(method="pearson").iloc[0, 1]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Scatterplot
ax.scatter(df_merged["sum_weights"], df_merged["median_height"], alpha=0.6)

# Regression line
mask = df_merged[["sum_weights", "median_height"]].dropna()
slope, intercept = np.polyfit(mask["sum_weights"], mask["median_height"], 1)
x_vals = np.array([mask["sum_weights"].min(), mask["sum_weights"].max()])
y_vals = slope * x_vals + intercept
ax.plot(x_vals, y_vals, color="blue")

# Correlation coefficient annotation
ax.text(20, 70, f"pearson r2 = {round(correlation_SR, 2)}",
        color="black", fontsize=12, ha="left")

# Labels and title
ax.set_xlabel("PRS")
ax.set_ylabel("Self Report + Exam")
ax.set_title("Height vs Height Risk Scores")

plt.tight_layout()
plt.show()

In [ ]:
# Calculate summary stats for each sex and each decile
df_summary_split = (
    df_merged.groupby(["sum_weights_decile", "sex_at_birth"])["median_height"]
    .agg(mean_height="mean", sem=lambda x: x.std() / np.sqrt(x.count()))
    .reset_index()
)

# Plot
color_map = {"Male": "#6E788D", "Female": "#C5373D"}
fill_map = {"Male": "#96A0B3", "Female": "#DC6464"}

fig, ax = plt.subplots(figsize=(9, 7))

for sex, group in df_summary_split.groupby("sex_at_birth"):
    group = group.sort_values("sum_weights_decile")

    # Line
    ax.plot(group["sum_weights_decile"], group["mean_height"],
            color=color_map[sex], linewidth=2.5, label=sex, zorder=2)

    # Points (white edge, colored fill)
    ax.scatter(group["sum_weights_decile"], group["mean_height"],
               facecolor=fill_map[sex], edgecolor="white", linewidth=1.5,
               s=80, zorder=3)

    # Error bars
    ax.errorbar(group["sum_weights_decile"], group["mean_height"],
            yerr=group["sem"], fmt="none", ecolor="black",
            capsize=4, elinewidth=1.2, zorder=5)  # bump above markers

ax.set_xticks(range(1, 11))

ax.set_xlabel("Height Polygenic Risk Score Decile", fontsize=16, fontweight="bold", labelpad=20)
ax.set_ylabel("Height (cm), mean (SE)", fontsize=16, fontweight="bold", labelpad=20)

ax.tick_params(axis="both", labelsize=14, color="black", width=1.2, length=0.25 * 72)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontweight("bold")
    label.set_color("black")

# Classic theme: only bottom/left spines visible
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(1.0)
ax.spines["bottom"].set_linewidth(1.0)
ax.spines["left"].set_color("black")
ax.spines["bottom"].set_color("black")

# Legend
legend = ax.legend(
    title="Sex at Birth", loc="upper left", bbox_to_anchor=(0.05, 0.99),
    frameon=True, fontsize=12, title_fontsize=14
)
legend.get_frame().set_facecolor("white")
legend.get_frame().set_edgecolor("black")
for text in legend.get_texts():
    text.set_fontweight("bold")
legend.get_title().set_fontweight("bold")

plt.tight_layout()
plt.savefig("height_by_prs_decile_sex.pdf", format="pdf", bbox_inches="tight")
plt.savefig("height_by_prs_decile_sex.png", format="png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Decile split on height
d1_cutoff = df_merged["median_height"].quantile(0.10)
d10_cutoff = df_merged["median_height"].quantile(0.90)

df_extremes = df_merged[
    (df_merged["median_height"] <= d1_cutoff) | (df_merged["median_height"] >= d10_cutoff)
].copy()

df_extremes["height_decile"] = np.where(
    df_extremes["median_height"] <= d1_cutoff, "Bottom Decile", "Top Decile"
)

color_map = {"Bottom Decile": "#E9D0E7", "Top Decile": "#7F3B7A"}

fig, ax = plt.subplots(figsize=(3.2, 2.6))

for decile, group in df_extremes.groupby("height_decile"):
    x = group["sum_weights"].dropna()  # PRS distribution, split by height decile
    mu, sigma = x.mean(), x.std()
    xs = np.linspace(mu - 4*sigma, mu + 4*sigma, 200)
    ys = norm.pdf(xs, mu, sigma)
    ax.plot(xs, ys, color=color_map[decile], linewidth=3)
    ax.axvline(mu, color=color_map[decile], linewidth=3, linestyle=(0, (4, 3)))

ax.set_xlabel("Genetic Risk", fontsize=24, color="black")
ax.set_ylabel("Density", fontsize=24, color="black")
ax.set_xticks([])
ax.set_yticks([])

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(2)
ax.spines["bottom"].set_linewidth(2)
ax.spines["left"].set_color("black")
ax.spines["bottom"].set_color("black")

plt.tight_layout()
plt.savefig("prs_by_height_top_bottom_deciles.pdf", format="pdf", dpi=300, bbox_inches="tight")
plt.show()